# Trabalho G2 — Ciência de Dados

## Objetivo

Aplicar sobre o conjunto de dados real de vendas em feiras (`dataset.csv`) o fluxo completo visto em sala:

1. **Pré-processamento e Engenharia de Recursos (preparar os dados):**
   - Limpar os dados de vendas das feiras e criar variáveis novas.
   - Demonstrar na prática as técnicas vistas em aula (outliers, imputação de nulos, discretização, encoding, correção de assimetria, escalonamento, seleção de variáveis e PCA) — *mesmo as que o modelo final não precisa*.
   - Empacotar o pré-processamento real em um `Pipeline` + `ColumnTransformer` (sem *data leakage*).

2. **Treino, Otimização e Avaliação de Modelos (aplicar ML):**
   - Separar treino e teste antes de qualquer ajuste.
   - Rodar um baseline e comparar classificadores vistos em aula (**Regressão Logística**, **KNN**, **Random Forest** e **XGBoost**) com validação cruzada e ajuste de hiperparâmetros (`GridSearchCV`).
   - Avaliar no teste (métricas, Matriz de Confusão, Curva ROC), validar o modelo campeão com **Nested CV**, explicar o resultado com **SHAP** e salvar o pipeline final (`joblib`).

> **Problema definido:** Classificação Binária do meio de pagamento no nível do pedido — prever **`pagamento_digital`** (`1` = `Cartão` ou `PIX`; `0` = `Dinheiro`) para apoiar o planejamento operacional de troco, maquininhas e conectividade nas feiras.

## Etapa 1 — Carregamento, Inspeção Inicial e Limpeza dos Dados

Nesta etapa carregamos o `dataset.csv` bruto (1 linha por item vendido), auditamos sua estrutura e corrigimos ruídos operacionais do sistema de caixa antes de consolidar os pedidos.

In [1]:
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

df_raw = pd.read_csv("dataset.csv")

print(f"Dimensões brutas: {df_raw.shape[0]} linhas x {df_raw.shape[1]} colunas")
print(f"Pedidos únicos  : {df_raw['pedido_id'].nunique()}")
print("\n--- Estrutura (.info) ---")
df_raw.info()

print("\n--- Valores nulos por coluna ---")
display(df_raw.isnull().sum().to_frame("nulos"))

print("\n--- Resumo estatístico (numéricas) ---")
display(df_raw.describe().round(2))

Dimensões brutas: 3815 linhas x 13 colunas
Pedidos únicos  : 3003

--- Estrutura (.info) ---
<class 'pandas.DataFrame'>
RangeIndex: 3815 entries, 0 to 3814
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   pedido_id                   3815 non-null   str    
 1   pedido_valor_total          3815 non-null   float64
 2   pedido_data                 3815 non-null   str    
 3   pedido_tipo_desconto        3815 non-null   str    
 4   pedido_valor_desconto       3815 non-null   float64
 5   pedido_subtotal             3815 non-null   int64  
 6   pedido_vendedor             3814 non-null   str    
 7   pedido_metodo_pagamento     3815 non-null   str    
 8   item_pedido_nome_produto    3815 non-null   str    
 9   item_pedido_quantidade      3815 non-null   int64  
 10  item_pedido_preco_unitario  3815 non-null   float64
 11  item_pedido_user_id         3814 non-null   str    
 12  evento_n

,nulos
pedido_id,0
pedido_valor_total,0
pedido_data,0
pedido_tipo_desconto,0
pedido_valor_desconto,0
pedido_subtotal,0
pedido_vendedor,1
pedido_metodo_pagamento,0
item_pedido_nome_produto,0
item_pedido_quantidade,0



--- Resumo estatístico (numéricas) ---


,pedido_valor_total,pedido_valor_desconto,pedido_subtotal,item_pedido_quantidade,item_pedido_preco_unitario
count,3815.00,3815.00,3815.00,3815.00,3815.00
mean,44.62,0.85,45.48,2.52,13.88
std,190.28,21.10,191.62,7.73,9.18
min,0.00,0.00,5.00,0.00,4.00
25%,15.00,0.00,15.00,1.00,6.00
50%,25.00,0.00,25.00,1.00,10.00
75%,40.00,0.00,40.00,2.00,25.00
max,3693.00,915.00,3696.00,348.00,80.00


### 1.1 Remoção de colunas irrelevantes, auditoria de descontos e itens cancelados

- **`pedido_vendedor` e `item_pedido_user_id`:** são 100% idênticas, contêm apenas 2 UUIDs de login compartilhado de caixa (além do único `NaN` da base) e não representam clientes nem vendedores individuais $\rightarrow$ removidas.
- **`pedido_tipo_desconto`:** antes de descartá-la, comprovamos matematicamente se `subtotal - desconto == valor_total` em 100% das linhas (desconto já gravado em R$).
- **`item_pedido_quantidade <= 0`:** removemos itens cancelados/zerados dentro de pedidos válidos.

In [2]:
df = df_raw.copy()

# 1. Confirmar redundância e remover colunas de login genérico de caixa
assert (df["pedido_vendedor"].fillna("NA") == df["item_pedido_user_id"].fillna("NA")).all()
df = df.drop(columns=["pedido_vendedor", "item_pedido_user_id"])

# 2. Validar matematicamente que 100% dos descontos já estão em moeda (R$)
erro_desconto = (df["pedido_subtotal"] - df["pedido_valor_desconto"] - df["pedido_valor_total"]).abs()
assert (erro_desconto < 0.01).all(), "Há divergência na conta de descontos!"
print(f"Auditoria de descontos OK — erro máximo: R$ {erro_desconto.max():.6f} (100% em R$).")
df = df.drop(columns=["pedido_tipo_desconto"])

# 3. Remover itens com quantidade não positiva (itens cancelados)
n_antes = len(df)
df = df[df["item_pedido_quantidade"] > 0].copy()
n_removidos = n_antes - len(df)

assert n_removidos == 2
assert len(df) == 3813
assert df["pedido_id"].nunique() == 3003

print(f"Itens com qtd <= 0 removidos: {n_removidos} | Base atual: {len(df)} linhas e {df['pedido_id'].nunique()} pedidos.")

Auditoria de descontos OK — erro máximo: R$ 0.000000 (100% em R$).
Itens com qtd <= 0 removidos: 2 | Base atual: 3813 linhas e 3003 pedidos.


### 1.2 Padronização de textos, feiras e meios de pagamento

- Removemos espaços sobrando em nomes de produtos e eventos.
- Unificamos os dois caixas da mesma feira: `"1 Moranguinho"` e `"2-Moranguinho"` $\rightarrow$ `"Festa do Moranguinho"`.
- Padronizamos os meios de pagamento: `"Pix"`/`"PIX"` $\rightarrow$ `"PIX"`, e `"Cartão de Crédito"`/`"Cartão de Débito"` $\rightarrow$ `"Cartão"` (armazenado em `pagamento_grupo`).

In [3]:
def normalizar_texto(texto: str) -> str:
    return re.sub(r"\s+", " ", str(texto)).strip()

df["item_pedido_nome_produto"] = df["item_pedido_nome_produto"].apply(normalizar_texto)
df["evento_nome"] = df["evento_nome"].apply(normalizar_texto).replace({
    "1 Moranguinho": "Festa do Moranguinho",
    "2-Moranguinho": "Festa do Moranguinho",
})

df["pagamento_grupo"] = (
    df["pedido_metodo_pagamento"]
    .apply(normalizar_texto)
    .replace({
        "Pix": "PIX",
        "Cartão de Crédito": "Cartão",
        "Cartão de Débito": "Cartão",
    })
)

assert set(df["pagamento_grupo"].unique()) == {"Dinheiro", "Cartão", "PIX"}

print("Eventos padronizados (linhas de itens):")
display(df["evento_nome"].value_counts().to_frame("itens"))

print("Meios de pagamento agrupados (linhas de itens):")
display(df["pagamento_grupo"].value_counts().to_frame("itens"))

Eventos padronizados (linhas de itens):


,itens
evento_nome,
Expointer_2026,705
Expo Afubra,610
Festa do Moranguinho,462
Wolksfest Agudo,424
ExpoDireto,405
EXPOBENTO,303
ExpoRestinga,236
FENAROZ,203
Expo Silveira Martins,173


Meios de pagamento agrupados (linhas de itens):


,itens
pagamento_grupo,
Dinheiro,2197
Cartão,1209
PIX,407


### 1.3 Extração de produto base, família comercial e multiplicador de embalagem

Padronizamos os `58` nomes brutos de produtos em:
- **`produto_base`** (`20` produtos canônicos, removendo texto entre parênteses e unificando o sinônimo `"Amendoim salgado e temperado"` $\rightarrow$ `"Amendoim Temperado"`).
- **`familia_produto`** (`6` famílias comerciais: `Cuca`, `Rapadura`, `Alfajor`, `Amendoim_Doces`, `Bolacha` e `Pao`).
- **`tipo_embalagem` e `multiplicador`** (quantidade de unidades em cada pacote/fardo/caixa, incluindo os `6` itens sem número explícito nos parênteses e a exceção verificada por preço de `Rapadura de Melado Grão Moído (Pacote x1)` a R$ 20,00 $\rightarrow$ `m = 3`).

In [4]:
def extrair_produto_base(nome: str) -> str:
    base = re.sub(r"\s*\([^)]*\)\s*", " ", nome)
    base = re.sub(r"\s+", " ", base).strip()
    if base == "Amendoim salgado e temperado":
        return "Amendoim Temperado"
    return base

def mapear_familia(produto_base: str) -> str:
    nl = produto_base.lower()
    if "cuca" in nl:
        return "Cuca"
    if "rapadura" in nl:
        return "Rapadura"
    if "alfajor" in nl:
        return "Alfajor"
    if any(k in nl for k in ["amendoim", "cri-cri", "carrapinha", "pé-de-moleque"]):
        return "Amendoim_Doces"
    if "bolacha" in nl:
        return "Bolacha"
    if "pão" in nl:
        return "Pao"
    return "Outros"

MAPA_SEM_MULTIPLICADOR = {
    ("Cuca Alemã", "Fardo"): 3,
    ("Rapadura Assada", "Pacote"): 6,
    ("Rapadura de Melado", "Pacote"): 3,
    ("Alfajor Preto", "Pacote"): 4,
    ("Cuca Enrolada de Amendoim", "Pacote"): 2,
    ("Cuca Enrolada de Chocolate", "Pacote"): 2,
}

def extrair_embalagem_e_multiplicador(row) -> tuple[str, int]:
    nome = row["item_pedido_nome_produto"]
    base = row["produto_base"]
    preco = row["item_pedido_preco_unitario"]

    m_expl = re.search(r"\((Pacote|Fardo|Bandeja|Caixa|Display)\s+x(\d+)\)\s*$", nome, re.I)
    if m_expl:
        emb = m_expl.group(1).capitalize()
        mult = int(m_expl.group(2))
        # Exceção verificada por preço: pacote vendido a R$ 20 contém 3 unidades (R$ 6,67/un)
        if base == "Rapadura de Melado Grão Moído" and emb == "Pacote" and mult == 1 and preco >= 18.0:
            return "Pacote", 3
        return emb, mult

    m_sem = re.search(r"\((Pacote|Fardo)\)\s*$", nome, re.I)
    if m_sem:
        emb = m_sem.group(1).capitalize()
        return emb, MAPA_SEM_MULTIPLICADOR[(base, emb)]

    return "Unidade", 1

df["produto_base"] = df["item_pedido_nome_produto"].apply(extrair_produto_base)
df["familia_produto"] = df["produto_base"].apply(mapear_familia)

emb_mult = df.apply(extrair_embalagem_e_multiplicador, axis=1)
df["tipo_embalagem"] = [x[0] for x in emb_mult]
df["multiplicador"] = [x[1] for x in emb_mult]

assert df["produto_base"].nunique() == 20
assert (df["familia_produto"] != "Outros").all()

print(f"Produtos base únicos: {df['produto_base'].nunique()} | Famílias: {df['familia_produto'].nunique()}")
display(
    df.groupby("familia_produto")
    .agg(itens=("pedido_id", "count"), produtos_base=("produto_base", "nunique"))
    .sort_values("itens", ascending=False)
)

Produtos base únicos: 20 | Famílias: 6


,itens,produtos_base
familia_produto,,
Cuca,1639,3
Rapadura,1184,3
Alfajor,569,2
Amendoim_Doces,308,5
Bolacha,104,4
Pao,9,3


### 1.4 Reconciliação contábil de quantidades e preços de pacotes

Em alguns registros (principalmente na *ExpoDireto*), o caixa gravou a quantidade em **unidades individuais**, mas registrou o **preço do pacote inteiro** em `item_pedido_preco_unitario`, gerando divergência entre `sum(quantidade * preco)` e `pedido_subtotal`.

Reconciliamos todos os itens em 4 casos mutuamente exclusivos e validamos com `assert` que a soma reconciliada bate exatamente com o subtotal de todos os `3.003` pedidos.

In [5]:
# 1. Identificar pedidos contabilmente divergentes antes da reconciliação
df["_valor_bruto"] = df["item_pedido_quantidade"] * df["item_pedido_preco_unitario"]
soma_bruta = df.groupby("pedido_id")["_valor_bruto"].sum()
subtotal_ped = df.groupby("pedido_id")["pedido_subtotal"].first()
pedidos_divergentes = set(soma_bruta[(soma_bruta - subtotal_ped).abs() >= 0.01].index)
df["_divergente"] = df["pedido_id"].isin(pedidos_divergentes)

print(f"Pedidos divergentes antes da reconciliação: {len(pedidos_divergentes)}")

# 2. Reconciliação (Casos A, B, C e D)
def reconciliar_item(row) -> tuple[float, float, float]:
    q = row["item_pedido_quantidade"]
    p = row["item_pedido_preco_unitario"]
    m = row["multiplicador"]
    nome = row["item_pedido_nome_produto"]
    reg_pacotes = (
        (m > 1)
        and (not row["_divergente"])
        and (
            row["evento_nome"] == "ExpoDireto"
            or "(Pacote)" in nome
            or "(Fardo)" in nome
            or "(Pacote x1)" in nome
        )
    )
    if m == 1:                  # Caso A: venda unitária
        return float(q), float(q), float(p)
    elif row["_divergente"]:    # Caso B: qtd em unidades + preço do pacote (ExpoDireto)
        return float(q), float(q / m), float(p / m)
    elif reg_pacotes:           # Caso C: qtd em pacotes + preço do pacote
        return float(q * m), float(q), float(p / m)
    else:                       # Caso D: qtd em unidades + preço unitário (demais feiras)
        return float(q), float(q / m), float(p)

rec = df.apply(reconciliar_item, axis=1)
df["qtd_unidades"] = [x[0] for x in rec]
df["qtd_pacotes"] = [x[1] for x in rec]
df["preco_unitario"] = [x[2] for x in rec]
df["item_valor_total"] = (df["qtd_unidades"] * df["preco_unitario"]).round(6)

# 3. Provas reais contábeis (asserts)
erro_pacotes = (df["qtd_pacotes"] - df["qtd_pacotes"].round()).abs().max()
assert erro_pacotes <= 1e-6, f"Foram encontrados pacotes fracionários: {erro_pacotes}"

soma_reconciliada = df.groupby("pedido_id")["item_valor_total"].sum()
erro_contabil_max = (soma_reconciliada - subtotal_ped).abs().max()
n_divergentes_pos = ((soma_reconciliada - subtotal_ped).abs() >= 0.01).sum()

assert n_divergentes_pos == 0 and erro_contabil_max < 0.01

df = df.drop(columns=["_valor_bruto", "_divergente"])
print(f"Pedidos divergentes após reconciliação  : {n_divergentes_pos}")
print(f"Erro absoluto máximo vs. pedido_subtotal: R$ {erro_contabil_max:.6f}")

Pedidos divergentes antes da reconciliação: 139
Pedidos divergentes após reconciliação  : 0
Erro absoluto máximo vs. pedido_subtotal: R$ 0.000000


## Etapa 2 — Consolidação por Pedido e Engenharia de Recursos (Variáveis Novas)

Agora transformamos a tabela de itens (`3.813` linhas) em **1 linha por pedido (`3.003` pedidos)** e construímos o alvo binário `pagamento_digital` junto com as novas variáveis de cesta, tempo e famílias de produto.

In [6]:
# 1. Consolidar itens no nível do pedido
df_pedidos = df.groupby("pedido_id", as_index=False).agg(
    pedido_data=("pedido_data", "first"),
    evento_nome=("evento_nome", "first"),
    pagamento_grupo=("pagamento_grupo", "first"),
    pedido_valor_total=("pedido_valor_total", "first"),
    pedido_subtotal=("pedido_subtotal", "first"),
    pedido_valor_desconto=("pedido_valor_desconto", "first"),
    qtd_itens=("item_pedido_nome_produto", "count"),
    qtd_unidades=("qtd_unidades", "sum"),
    eh_pacote=("multiplicador", lambda s: int((s > 1).any())),
)

# 2. Features derivadas de desconto e criação do alvo binário (y)
df_pedidos["teve_desconto"] = (df_pedidos["pedido_valor_desconto"] > 0).astype(int)
df_pedidos["pagamento_digital"] = (df_pedidos["pagamento_grupo"] != "Dinheiro").astype(int)

print("Distribuição de pagamento_grupo (3.003 pedidos):")
display(
    pd.DataFrame({
        "pedidos": df_pedidos["pagamento_grupo"].value_counts(),
        "pct (%)": (df_pedidos["pagamento_grupo"].value_counts(normalize=True) * 100).round(2),
    })
)

# Remover colunas originais de pagamento/desconto redundante para evitar data leakage
df_pedidos = df_pedidos.drop(columns=["pagamento_grupo", "pedido_valor_desconto"])

Distribuição de pagamento_grupo (3.003 pedidos):


,pedidos,pct (%)
pagamento_grupo,,
Dinheiro,1791,59.64
Cartão,908,30.24
PIX,304,10.12


### 2.1 Variáveis Temporais (fuso horário, fim de semana, horário cíclico e turno)

Extraímos da data da venda (`America/Sao_Paulo`):
- `hora` e `dia_semana` *(mantida temporariamente para checagem de correlação com fim de semana na Etapa 3)*;
- `eh_fim_de_semana` (`1` = sábado ou domingo);
- **Horário cíclico** (`hora_sin` e `hora_cos`), preservando a continuidade entre `23h` e `00h`;
- **`turno`** (discretização personalizada por regra de negócio: `Manhã`, `Tarde` e `Noite`).

In [7]:
dt_sp = pd.to_datetime(df_pedidos["pedido_data"], format="mixed", utc=True).dt.tz_convert("America/Sao_Paulo")

df_pedidos["hora"] = dt_sp.dt.hour
df_pedidos["dia_semana"] = dt_sp.dt.day_of_week
df_pedidos["eh_fim_de_semana"] = (df_pedidos["dia_semana"] >= 5).astype(int)

# Codificação cíclica da hora (período = 24h)
df_pedidos["hora_sin"] = np.sin(2 * np.pi * df_pedidos["hora"] / 24)
df_pedidos["hora_cos"] = np.cos(2 * np.pi * df_pedidos["hora"] / 24)

# Discretização personalizada em turnos
df_pedidos["turno"] = pd.cut(
    df_pedidos["hora"],
    bins=[-1, 11, 17, 24],
    labels=["Manhã", "Tarde", "Noite"],
).astype(str)

df_pedidos = df_pedidos.drop(columns=["pedido_data"])

print("Distribuição por turno:")
display(df_pedidos["turno"].value_counts().to_frame("pedidos"))
print("Distribuição fim de semana (0=Dia útil, 1=Fim de semana):")
display(df_pedidos["eh_fim_de_semana"].value_counts().to_frame("pedidos"))

Distribuição por turno:


,pedidos
turno,
Tarde,2019
Noite,524
Manhã,460


Distribuição fim de semana (0=Dia útil, 1=Fim de semana):


,pedidos
eh_fim_de_semana,
0,1575
1,1428


### 2.2 Flags binárias das famílias de produto

Criamos uma variável binária (`0` ou `1`) para cada uma das `6` famílias comerciais presentes no pedido (`tem_cuca`, `tem_rapadura`, `tem_alfajor`, `tem_amendoim_doces`, `tem_bolacha` e `tem_pao`). A coluna `tem_pao` (~0,3% de frequência) é mantida aqui para ser diagnosticada e removida via `VarianceThreshold` na Etapa 3.

In [8]:
fam_flags = (pd.crosstab(df["pedido_id"], df["familia_produto"]) > 0).astype(int)

ordem_familias = ["Cuca", "Rapadura", "Alfajor", "Amendoim_Doces", "Bolacha", "Pao"]
fam_flags = fam_flags[ordem_familias].rename(
    columns={f: f"tem_{f.lower()}" for f in ordem_familias}
)

df_pedidos = df_pedidos.merge(fam_flags, on="pedido_id", how="left")

cols_fam = [f"tem_{f.lower()}" for f in ordem_familias]
resumo_fam = pd.DataFrame({
    "pedidos": df_pedidos[cols_fam].sum(),
    "pct (%)": (df_pedidos[cols_fam].mean() * 100).round(2),
    "variancia": df_pedidos[cols_fam].var(ddof=0).round(4),
})
display(resumo_fam)

,pedidos,pct (%),variancia
tem_cuca,1541,51.32,0.2498
tem_rapadura,1065,35.46,0.2289
tem_alfajor,505,16.82,0.1399
tem_amendoim_doces,284,9.46,0.0856
tem_bolacha,101,3.36,0.0325
tem_pao,9,0.30,0.0030


### 2.3 Validação final da tabela consolidada

Conferimos a integridade completa da tabela `df_pedidos` (`3.003` pedidos únicos, `0` valores nulos, alvo binário limpo), pronta para a **Etapa 3** (diagnóstico/tratamento de outliers e demais técnicas de pré-processamento da aula).

In [9]:
colunas_ordenadas = [
    "pedido_id",
    "evento_nome",
    "pedido_valor_total",
    "pedido_subtotal",
    "qtd_itens",
    "qtd_unidades",
    "eh_pacote",
    "teve_desconto",
    "hora",
    "dia_semana",
    "eh_fim_de_semana",
    "hora_sin",
    "hora_cos",
    "turno",
    "tem_cuca",
    "tem_rapadura",
    "tem_alfajor",
    "tem_amendoim_doces",
    "tem_bolacha",
    "tem_pao",
    "pagamento_digital",
]
df_pedidos = df_pedidos[colunas_ordenadas]

# Asserts finais de integridade
assert len(df_pedidos) == 3003
assert df_pedidos["pedido_id"].nunique() == 3003
assert df_pedidos.isna().sum().sum() == 0
assert set(df_pedidos["pagamento_digital"].unique()) == {0, 1}
assert "pagamento_grupo" not in df_pedidos.columns
assert "pedido_metodo_pagamento" not in df_pedidos.columns

print(f"Base consolidada OK: {df_pedidos.shape[0]} pedidos x {df_pedidos.shape[1]} colunas | Nulos: {df_pedidos.isna().sum().sum()}")
print("\nDistribuição do alvo (pagamento_digital: 0=Dinheiro, 1=Digital):")
display(
    pd.DataFrame({
        "pedidos": df_pedidos["pagamento_digital"].value_counts().sort_index(),
        "pct (%)": (df_pedidos["pagamento_digital"].value_counts(normalize=True).sort_index() * 100).round(2),
    })
)

display(df_pedidos.head())

Base consolidada OK: 3003 pedidos x 21 colunas | Nulos: 0

Distribuição do alvo (pagamento_digital: 0=Dinheiro, 1=Digital):


,pedidos,pct (%)
pagamento_digital,,
0,1791,59.64
1,1212,40.36


,pedido_id,evento_nome,pedido_valor_total,pedido_subtotal,qtd_itens,qtd_unidades,eh_pacote,teve_desconto,hora,dia_semana,eh_fim_de_semana,hora_sin,hora_cos,turno,tem_cuca,tem_rapadura,tem_alfajor,tem_amendoim_doces,tem_bolacha,tem_pao,pagamento_digital
0,0055b17b-e0ca-4884-afc9-309823f6e500,Expointer_2026,15.0,15,1,1.0,0,0,19,3,0,-0.965926,0.258819,Noite,1,0,0,0,0,0,0
1,008eadab-2a7c-46b0-90de-ca9c7fb7c3ab,Expointer_2026,100.0,100,1,4.0,0,0,17,2,0,-0.965926,-0.258819,Tarde,1,0,0,0,0,0,1
2,009fd014-f883-44bc-a444-c134345e91b3,Expo Afubra,15.0,15,1,1.0,0,0,13,2,0,-0.258819,-0.965926,Tarde,1,0,0,0,0,0,1
3,00b6a066-e32d-4704-ab6e-167895bcb7cb,ExpoDireto,6.0,6,1,1.0,0,0,13,3,0,-0.258819,-0.965926,Tarde,0,0,1,0,0,0,1
4,00d1ef1f-84fd-4b4f-9b15-9e3161a56052,ExpoDireto,12.0,12,1,2.0,0,0,16,4,0,-0.866025,-0.500000,Tarde,0,0,1,0,0,0,0
